# 04 Embed And Pack Artifacts

Create BGE-M3 embedding JSONL artifacts, run offline retrieval smoke, and zip outputs for download.

In [ ]:
import os, shutil, subprocess, sys
from pathlib import Path

PROJECT_DIR = Path(os.environ.get('PROJECT_DIR', '/kaggle/input/tuvi-battu-graphrag/tuvi-battu-graphrag'))
if not PROJECT_DIR.exists():
    PROJECT_DIR = Path.cwd()
OUTPUT_ROOT = Path('/kaggle/working/w3_local_outputs')
os.environ['PYTHONDONTWRITEBYTECODE'] = '1'
REPORTS = OUTPUT_ROOT / 'reports'
STATE = OUTPUT_ROOT / 'state'
SOURCES = ['TVKL', 'TVNL', 'TVHS', 'TVGM']
STRATEGIES = ['chunk_fixed_512', 'chunk_structure_parent_child', 'chunk_semantic_embedding_bge_m3']


In [ ]:
for strategy in STRATEGIES:
    for source in SOURCES:
        cmd = [
            sys.executable, '-B', str(PROJECT_DIR / 'scripts' / 'embed_chunks.py'),
            '--chunks-input', str(OUTPUT_ROOT / 'chunks' / strategy),
            '--source-id', source,
            '--chunking-strategy', strategy,
            '--output', str(OUTPUT_ROOT / 'embeddings' / strategy / f'{source}_{strategy}_embeddings.jsonl'),
            '--summary-output', str(REPORTS / f'embed_{source}_{strategy}.json'),
            '--retrieval-smoke-output', str(REPORTS / f'retrieval_{source}_{strategy}.json'),
            '--state-output', str(STATE / f'{source}_{strategy}_embedding_state.json'),
            '--resume',
            '--embedding-backend', 'local',
            '--model', 'BAAI/bge-m3',
            '--expected-dim', '1024',
            '--local-embedding-model', 'BAAI/bge-m3',
            '--local-embedding-batch-size', '16',
            '--vector-index-name', 'chunkVectorBgeM3',
        ]
        subprocess.run(cmd, cwd=PROJECT_DIR, check=True)


In [ ]:
archive = shutil.make_archive('/kaggle/working/w3_local_outputs', 'zip', OUTPUT_ROOT)
print('Wrote', archive)
